In [24]:
from pathlib import Path
import sys
import pandas as pd


PROJECT_ROOT = Path("..").resolve()

# Gör så att Python hittar src/
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_config

# Läs config
CONFIG_PATH = PROJECT_ROOT / "config.toml"
config = load_config(CONFIG_PATH, PROJECT_ROOT)


# Luftdata / PM2.5
air_df = pd.read_csv(
    config.raw_dir / "train_Malmö A.csv"
)

# Väderdata
weather_df = pd.read_csv(
    config.raw_dir / "train_Malmö Dalaplan.csv"
)

# Sammanfogad data
merged_df = pd.read_csv(
    config.merged_dir / "train_merged.csv"
)

## 1. Kontroll av struktur och datatyper

**Observation:**  
Samtliga mätvariabler är numeriska (`float64`). Tidsvariabeln `tid` är däremot inläst som `str` och behöver konverteras till `datetime` innan vidare analys av tidsserien.

Eftersom tidsstämplarna innehåller både svensk sommartid (CEST) och vintertid (CET) konverteras `tid` i denna notebook till timezone-aware `datetime` (`Europe/Stockholm`). Detta görs här för att möjliggöra korrekt analys av tidsserien. I den slutliga pipelinen bör konverteringen i stället hanteras vid inläsning av data.

In [33]:
print("LUFTDATA")
air_df.info()

print("\nVÄDERDATA")
weather_df.info()

print("\nMERGED DATA")
merged_df.info()

LUFTDATA
<class 'pandas.DataFrame'>
RangeIndex: 22290 entries, 0 to 22289
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   tid                    22290 non-null  str    
 1   Lufttemperatur         22289 non-null  float64
 2   Vindriktning           22253 non-null  float64
 3   Vindhastighet          22253 non-null  float64
 4   Relativ luftfuktighet  22255 non-null  float64
 5   Nederbördsmängd        21756 non-null  float64
dtypes: float64(5), str(1)
memory usage: 1.0 MB

VÄDERDATA
<class 'pandas.DataFrame'>
RangeIndex: 22855 entries, 0 to 22854
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   tid     22855 non-null  str    
 1   pm25    22702 non-null  float64
dtypes: float64(1), str(1)
memory usage: 357.2 KB

MERGED DATA
<class 'pandas.DataFrame'>
RangeIndex: 22857 entries, 0 to 22856
Data columns (total 7 columns):
 #   Column   

In [38]:
for df in [air_df, weather_df, merged_df]:
    df["tid"] = (pd.to_datetime(df["tid"], utc=True)
    .dt.tz_convert("Europe/Stockholm")
    )

print(air_df["tid"].dtype)
print(weather_df["tid"].dtype)
print(merged_df["tid"].dtype)

datetime64[us, Europe/Stockholm]
datetime64[us, Europe/Stockholm]
datetime64[us, Europe/Stockholm]


## 2. Kontroll av antal observationer och tidsintervall

**Förväntat antal observationer:** 22 824 timmar

**Observation:**
- PM2.5-data innehåller 22 290 observationer och följer det förväntade
  tidsintervallet 2023-05-26–2025-12-31.
- Väderdata innehåller 22 855 observationer, alltså fler rader än det
  förväntade antalet timmar. Dessutom sträcker sig tidsintervallet till
  2026-01-01 00:00, trots att träningsperioden slutar 2025-12-31.
- Merged data innehåller 22 857 observationer och sträcker sig också till
  2026-01-01 00:00.

Väderdata har hämtats om med nuvarande konfiguration och avvikelsen
kvarstår. Filtreringen av väderdata behöver därför kontrolleras,
framför allt hanteringen av start-/sluttid och tidszon.


In [34]:
config.train_start
config.train_slut

expected_timestamps = pd.date_range(
    start=config.train_start,
    end=f"{config.train_slut} 23:00:00",
    freq="h",
    tz="Europe/Stockholm"
)

expected_count = len(expected_timestamps)

print("Förväntat antal timmar:", expected_count)

print("Luftdata:", air_df.shape)
print("Väderdata:", weather_df.shape)
print("Merged:", merged_df.shape)

print("\nLUFTDATA")
print("Första:", air_df["tid"].min())
print("Sista:", air_df["tid"].max())

print("\nVÄDERDATA")
print("Första:", weather_df["tid"].min())
print("Sista:", weather_df["tid"].max())

print("\nMERGED")
print("Första:", merged_df["tid"].min())
print("Sista:", merged_df["tid"].max())

Förväntat antal timmar: 22825
Luftdata: (22290, 6)
Väderdata: (22855, 2)
Merged: (22857, 7)

LUFTDATA
Första: 2023-05-26 00:00:00+02:00
Sista: 2025-12-31 23:00:00+01:00

VÄDERDATA
Första: 2023-05-26 02:00:00+02:00
Sista: 2026-01-01 00:00:00+01:00

MERGED
Första: 2023-05-26 00:00:00+02:00
Sista: 2026-01-01 00:00:00+01:00


## 3. Kontroll av saknade värden

**Observation:**  
Merged data har en högre andel saknade värden än respektive ursprungsdata. 
Detta tyder på att dataseten delvis saknar observationer för olika tidsstämplar, 
vilket skapar ytterligare NaN-värden vid merge.

In [35]:
def missing_values(df):
    return pd.DataFrame({
        "Antal saknade": df.isna().sum(),
        "Andel saknade (%)": (df.isna().mean() * 100).round(2)
    })

print("\nLUFTDATA")
print(missing_values(air_df))

print("\nVÄDERDATA")
print(missing_values(weather_df))

print("\nMERGED DATA")
print(missing_values(merged_df))


LUFTDATA
                       Antal saknade  Andel saknade (%)
tid                                0               0.00
Lufttemperatur                     1               0.00
Vindriktning                      37               0.17
Vindhastighet                     37               0.17
Relativ luftfuktighet             35               0.16
Nederbördsmängd                  534               2.40

VÄDERDATA
      Antal saknade  Andel saknade (%)
tid               0               0.00
pm25            153               0.67

MERGED DATA
                       Antal saknade  Andel saknade (%)
tid                                0               0.00
pm25                             155               0.68
Lufttemperatur                   538               2.35
Vindriktning                     574               2.51
Vindhastighet                    574               2.51
Relativ luftfuktighet            572               2.50
Nederbördsmängd                 1074               4.70
